# 分類器


In [31]:
import os
import warnings
import numpy as np
import pandas as pd
import mne
import pymysql
from scipy.ndimage import uniform_filter1d
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# 關閉不必要的警告
warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level('WARNING')
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

In [32]:
PATH_TESTER = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/dataset_sorted/sample_test/test/tester'

DB_CONFIG = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': '',
    'database': 'eeg_db_sorted',
    'charset': 'utf8mb4'
}

# 特徵提取

# Alpha 頻帶

In [33]:
def calculate_paf(psd_values, freqs, alpha_range=(4, 12)):
    """計算 Alpha 頻帶峰值頻率 (PAF)"""
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    peak_idx = np.argmax(alpha_psd)
    return freqs[alpha_idx][peak_idx]

def calculate_cgf(psd_values, freqs, alpha_range=(4, 12)):
    """計算 Alpha 頻帶重心頻率 (CGF)"""
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    if np.sum(alpha_psd) == 0:
        return np.nan
    return np.sum(freqs[alpha_idx] * alpha_psd) / np.sum(alpha_psd)

def calculate_af_amplitude(psd_values, freqs, af):
    """計算指定頻率點的振幅值"""
    return np.interp(af, freqs, psd_values)


# HFD

In [34]:
def calculate_hfd(data, k_max=10):
    """計算單一通道時域訊號的 Higuchi Fractal Dimension (HFD)"""
    N = len(data)
    L_k = np.zeros(k_max)
    for k in range(1, k_max + 1):
        Lk_m = []
        for m in range(k):
            indices = np.arange(m, N, k)
            n_max = len(indices)
            if n_max > 1:
                diffs = np.abs(np.diff(data[indices]))
                l_m_k = (np.sum(diffs) * (N - 1) / ((n_max - 1) * k)) / k
                Lk_m.append(l_m_k)
        L_k[k-1] = np.mean(Lk_m) if Lk_m else 0
    valid = L_k > 0
    if not np.any(valid): 
        return np.nan
    x = np.log(1.0 / np.arange(1, k_max + 1)[valid])
    y = np.log(L_k[valid])
    hfd, _ = np.polyfit(x, y, 1)
    return hfd

# 讀取測試資料

In [35]:
def process_test_eeg_directory(path, event_name='Eyes Closed', delay_time=0, cut_time=8, k_max=10):
    """讀取測試資料夾內的所有 EDF 檔案並計算特徵"""
    if not os.path.exists(path):
        print(f"❌ 錯誤：找不到路徑 {path}")
        return pd.DataFrame()
        
    edf_files = [f for f in os.listdir(path) if f.lower().endswith('.edf')]
    test_features = []
    print(f"\n[開始處理測試集] 預計處理檔案數: {len(edf_files)}")
    
    for filename in edf_files:
        file_path = os.path.join(path, filename)
        try:
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        except Exception as e:
            print(f"  ❌ 檔案 {filename} 讀取失敗: {e}")
            continue

        try:
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            s_events = events[events[:, 2] == event_id[event_name]]
        except Exception as e:
            print(f"  ⚠️ 檔案 {filename} 未包含事件 '{event_name}': {e}")
            continue

        if 'Photic' in raw.info['ch_names']:
            raw.drop_channels(['Photic'])

        fs = int(raw.info['sfreq'])
        filter_low, filter_high = 0.5, 50
        ch_names = raw.info['ch_names']
        ch_types = ['eeg'] * len(ch_names)

        subject_psd_results = {ch: [] for ch in ch_names}
        subject_hfd_results = {ch: [] for ch in ch_names}
        freqs = None

        for s_event in s_events:
            event_start = s_event[0]
            start_sample = event_start + int(fs * delay_time)
            end_sample = start_sample + int(fs * cut_time)

            if end_sample > len(raw): 
                continue

            event_data, _ = raw[:, start_sample:end_sample]
            event_data_filtered = mne.filter.filter_data(event_data, sfreq=fs, l_freq=filter_low, h_freq=filter_high, verbose=False)
            
            info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=ch_types)
            epoch_raw = mne.io.RawArray(event_data_filtered, info, verbose=False)
            epoch_data_clean = epoch_raw.get_data()

            for ch_idx, ch_name in enumerate(ch_names):
                psd, freqs = mne.time_frequency.psd_array_welch(epoch_data_clean[ch_idx], 
                                                                sfreq=fs, fmin=filter_low, fmax=filter_high, 
                                                                n_fft=int(fs*2), verbose=False)
                psd = psd * (1e6 ** 2)
                subject_psd_results[ch_name].append(psd)
                
                hfd_val = calculate_hfd(epoch_data_clean[ch_idx], k_max=k_max)
                subject_hfd_results[ch_name].append(hfd_val)

        if len(subject_psd_results[ch_names[0]]) == 0:
            print(f"  ⚠️ 檔案 {filename} 無有效 Epoch 區段")
            continue

        patient_id = os.path.splitext(filename)[0]
        feature_dict = {'Filename': filename, 'patient_id': patient_id}
        
        # 儲存通道層級 HFD 平均
        all_hfd_values = []
        for ch_name in ch_names:
            hfd_avg = np.mean(subject_hfd_results[ch_name])
            feature_dict[f'HFD_{ch_name}'] = hfd_avg
            all_hfd_values.append(hfd_avg)
            
        # 全通道 PSD 與 Alpha 指標計算
        all_channel_psds = [np.mean(subject_psd_results[ch], axis=0) for ch in ch_names]
        global_mean_psd_linear = np.mean(all_channel_psds, axis=0)
        global_mean_psd_db = 10 * np.log10(global_mean_psd_linear)
        
        alpha_band_idx = [i for i, f in enumerate(freqs) if 8 <= f < 12]
        feature_dict['alpha_average_power'] = np.mean(global_mean_psd_db[alpha_band_idx])
        feature_dict['alpha_PAF'] = calculate_paf(global_mean_psd_linear, freqs)
        feature_dict['alpha_CAF'] = calculate_cgf(global_mean_psd_linear, freqs)
        feature_dict['hfd_mean'] = np.mean(all_hfd_values)

        test_features.append(feature_dict)
        
    return pd.DataFrame(test_features)

# 餘弦相似度計算

In [36]:
# 4. 資料庫對接與特徵載入
def load_db_features():
    """從 MySQL 資料庫中載入 alpha_band, hfd 與 patients 表格之特徵"""
    conn = pymysql.connect(**DB_CONFIG)
    
    query = """
    SELECT 
        p.patient_id,
        p.group_id,
        a.average_power AS alpha_average_power,
        a.PAF AS alpha_PAF,
        a.CAF AS alpha_CAF,
        h.Fp1_F3, h.F3_C3, h.C3_P3, h.P3_O1, h.Fp2_F4, h.F4_C4, h.C4_P4, h.P4_O2,
        h.Fp1_F7, h.F7_T3, h.T3_T5, h.T5_O1, h.Fp2_F8, h.F8_T4, h.T4_T6, h.T6_O2,
        h.Fz_Cz, h.Cz_Pz
    FROM patients p
    JOIN alpha_band a ON p.patient_id = a.patient_id
    JOIN hfd h ON p.patient_id = h.patient_id;
    """
    
    # 使用 with 語句確保連線一定會關閉
    with conn:
        df_db = pd.read_sql(query, conn)
    
    # 動態取得所有 HFD 通道的欄位名稱
    exclude_cols = ['patient_id', 'group_id', 'alpha_average_power', 'alpha_PAF', 'alpha_CAF']
    hfd_cols = [col for col in df_db.columns if col not in exclude_cols]
    
    return df_db, hfd_cols

# 5. 主程式流程：計算相似度並輸出前十筆最相似資料
if __name__ == "__main__":
    # A. 提取測試集特徵
    df_test = process_test_eeg_directory(PATH_TESTER)
    
    if df_test.empty:
        print("❌ 無法提取測試集特徵，程序結束。")
        exit()
    
    # 清理 df_test 的欄位名稱：移除前綴並將 '-' 轉為 '_'
    df_test.columns = df_test.columns.str.replace('HFD_EEG ', '', regex=False).str.replace('-', '_', regex=False)

    # B. 從 MySQL 載入歷史資料庫特徵
    try:
        df_db, hfd_cols = load_db_features()
        print(f"\n[資料庫連接成功] 共載入 {len(df_db)} 筆歷史資料庫對照特徵")
    except Exception as e:
        print(f"❌ 資料庫連接或查詢失敗: {e}")
        exit()
        
    # C. 定義特徵欄位
    alpha_features = ['alpha_average_power', 'alpha_PAF', 'alpha_CAF']

    # 防呆檢查：確認 df_test 包含所有必要的欄位
    missing_cols = set(alpha_features + hfd_cols) - set(df_test.columns)
    if missing_cols:
        print(f"❌ 測試資料缺少以下必要欄位: {missing_cols}")
        exit()

    # D. 提早準備資料庫的特徵矩陣 (提昇迴圈內的執行速度)
    X_db_alpha = df_db[alpha_features].values
    X_db_hfd = df_db[hfd_cols].values  # Shape: (N, 18) 通道 HFD 矩陣
    
    # E. 檢索比對
    print("\n" + "="*120)
    print(" 測試集與資料庫檢索比對結果 (以 Alpha 相似度排序前 10 筆，並顯示 HFD 綜合相似度)")
    print("="*120)

    for idx, test_row in df_test.iterrows():
        # 1. Alpha 頻帶綜合餘弦相似度 (1x3 vs Nx3)
        test_alpha_vector = test_row[alpha_features].values.reshape(1, -1)
        alpha_similarities = cosine_similarity(test_alpha_vector, X_db_alpha).flatten()
        
        # 2. HFD 全通道綜合餘弦相似度 (1x18 vs Nx18) -> 修正原本單數值比對的錯誤
        test_hfd_vector = test_row[hfd_cols].values.reshape(1, -1)
        hfd_similarities = cosine_similarity(test_hfd_vector, X_db_hfd).flatten()
        
        # 依 Alpha 相似度高低取得前 10 個索引
        top10_indices = np.argsort(alpha_similarities)[::-1][:10]
        
        patient_id_show = test_row.get('patient_id', 'Unknown')
        filename_show = test_row.get('Filename', f'Sample_{idx}')
        
        print(f"\n🔍 測試檔案: {filename_show} (受試者 ID: {patient_id_show})")
        print("-" * 120)
        print(f"{'排名':<6}{'Patient ID':<12}{'診斷組別':<12}{'Alpha 相似度':<15}{'HFD 綜合相似度':<15}")
        print("-" * 120)
        
        for rank, db_idx in enumerate(top10_indices, 1):
            matched_patient = df_db.iloc[db_idx]['patient_id']
            matched_group = df_db.iloc[db_idx]['group_id']
            a_sim = alpha_similarities[db_idx]
            h_sim = hfd_similarities[db_idx]
            
            print(f"{rank:<6}{matched_patient:<12}{matched_group:<12}{a_sim:<15.6f}{h_sim:<15.6f}")
            
        print("="*120)


[開始處理測試集] 預計處理檔案數: 1

[資料庫連接成功] 共載入 168 筆歷史資料庫對照特徵

 測試集與資料庫檢索比對結果 (以 Alpha 相似度排序前 10 筆，並顯示 HFD 綜合相似度)

🔍 測試檔案: A7729346.edf (受試者 ID: A7729346)
------------------------------------------------------------------------------------------------------------------------
排名    Patient ID  診斷組別        Alpha 相似度      HFD 綜合相似度      
------------------------------------------------------------------------------------------------------------------------
1     C2078682    MCI         0.995645       0.995394       
2     R5898147    MCI         0.995061       0.990457       
3     D0731134    AD          0.992852       0.993937       
4     C5036045    HC          0.992628       0.992193       
5     R1297941    HC          0.991921       0.994130       
6     A7668316    HC          0.991840       0.996412       
7     Q2479353    AD          0.991806       0.992991       
8     B9157951    AD          0.991707       0.991389       
9     C3742795    AD          0.991188       0.995612       
10 

C:\Users\kitt9\AppData\Local\Temp\ipykernel_2544\1009090663.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(query, conn)


# 